In [2]:
import pandas as pd
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

### Pregunta A (Sumarización Categórica): 
#### Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?

In [3]:
df['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

##### La tasa de supervivencia global del barco es de 38.38%

### Pregunta B (Agrupación y Agregación): 
##### El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

In [4]:
df.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

##### Sobrevivieron el 74.20% de las mujeres abordo, y solo el 18.89% de los hombres.


### Pregunta C: 
#### El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
#### Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
#### Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
#### Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

In [5]:
q1 = df['Fare'].quantile(.25)
q3 = df['Fare'].quantile(.75)

print(q1)
print(q3)


7.9104
31.0


In [6]:
iqr = q3 - q1

print(iqr)

23.0896


In [7]:
ul = q3 + 1.5 * iqr

print(ul)

65.6344


In [9]:
df_f = df[df['Fare'] > ul]
print(df_f['Pclass'].mean())

1.1637931034482758


##### En su mayoría pertenecían a primera clase.

### Pregunta D: 
#### Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

In [11]:
fare_mean = df['Fare'].mean()
fare_median = df['Fare'].median()

print(fare_mean)
print(fare_median)

32.204207968574636
14.4542


##### Se podría decir que hay un sesgo positivo y que el tercer cuartil está más disperso que el segundo. Podría afectar de manera que tendríamos outliers y quedarían aislados y esto tengo entendido también podría generar valles donde se creé se llegó a un resultado.

### Pregunta E: 
#### En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?

In [16]:

survived = round(150 * (df["Survived"] == 1).mean())
not_survived = 150 - survived

df_survived = df[df["Survived"] == 1].sample(survived, random_state=42)
df_not_survived = df[df["Survived"] == 0].sample(not_survived, random_state=42)

sample_150 = pd.concat([df_survived, df_not_survived])

sample_150.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,150.000000,150.000000,150.000000,116.000000,150.000000,150.000000,150.000000
mean,429.286667,0.386667,2.280000,30.698276,0.533333,0.460000,34.032722
std,268.329391,0.488618,0.852316,14.099627,1.185294,0.945707,48.720487
min,6.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,204.500000,0.000000,1.000000,22.000000,0.000000,0.000000,7.895800
50%,377.500000,0.000000,3.000000,29.000000,0.000000,0.000000,14.479150
75%,672.250000,1.000000,3.000000,39.250000,1.000000,1.000000,30.973950
max,886.000000,1.000000,3.000000,70.500000,8.000000,5.000000,262.375000


##### El sesgo de selección en el cual si extraemos una muestra pequeña en este de caso de 150 de 891 registros mediante un muestreo aleatorio simple, la varianza estadística puede seleccionar accidentalmente muchos menos sobrevivientes entonces ya no sería el mismo porcentaje de sobrevivientes que en la muestra completa.

### Pregunta F: 
#### Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

In [17]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

##### Creo que nuevamente de selección puesto que estamos ignorando datos, por ejemplo historicamente sabemos que los pasajeros de tercera clase eran más jóvenes, por ende al no tomar en cuenta, estos datos, puede dar la falsa impresión de que la edad de las personas que murieron era mayor.

### Pregunta G: 
#### Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.

##### No se deberían de eliminar las filas con .drop() puesto que hay modelos de machine learning que son sensibles a la magnitud y estaríamos perdiendo información crítica y creando un nuevo sesgo.

### Pregunta H: 
#### Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

##### La edad en el barco no es homogénea, entonces agrupar por título segmenta a los pasajeros por etapas reales. La diferencia entre la edad real y la mediana del grupo es mucho menor que respecto a una mediana global arbitraria

### Pregunta I: 
#### Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

In [19]:
survived_variance = df['Survived'].var()

print(survived_variance)

0.2367722165474984


##### Tendríamos problemas con la variabilidad puesto que no habría patrones a extraer, por ejemplo en este caso de la columna Survived una varianza = 0 hubiese querido decir que o todos los pasajeros hubiesen muerto o hubiesen sobrevivido.

### Pregunta J: 
#### Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

In [21]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

##### Ocurre un sobreajuste o overfitting, en el cual hay una memorización de ruido en lugar de señal, por ejemplo, si el único pasajero de un subgrupo específico, digamos una mujer de primera clase que abordó en Queenstown, sobrevivió, el modelo asignará una probabilidad de supervivencia de 100% a esa combinación exacta. Entonces asume como regla universal lo que en realidad fue una casualidad o una circunstancia aislada.